# ERA5 -> PRISM Downscaling Demo (Georgia)

## 1) Title + Goal
This notebook presents a compact research prototype for climate downscaling:
- input: coarse ERA5 daily 2m temperature
- target: higher-resolution PRISM daily temperature
- model: CNN downscaler baseline

Goal: establish a reproducible baseline before moving to stronger spatiotemporal models.


## 2) Data Explanation (ERA5 vs PRISM)
ERA5 provides broad, lower-resolution climate fields. PRISM provides finer regional temperature maps.

The learning task is supervised downscaling: for each matching date, map ERA5 to PRISM.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from IPython.display import Image, Markdown, display

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datasets.prism_dataset import ERA5_PRISM_Dataset
from models.cnn_downscaler import CNNDownscaler


In [ ]:
ERA5_PATH = ROOT / 'data_raw' / 'era5_georgia_temp.nc'
PRISM_PATH = ROOT / 'data_raw' / 'prism'
CHECKPOINT_PATH = ROOT / 'checkpoints' / 'cnn_downscaler_best.pt'
RESULTS_DIR = ROOT / 'results' / 'evaluation'

print('ERA5:', ERA5_PATH)
print('PRISM:', PRISM_PATH)
print('Checkpoint:', CHECKPOINT_PATH)
print('Results:', RESULTS_DIR)


## 3) Model Overview (CNN Downscaler)
The baseline model uses a small convolutional encoder-decoder:
- encoder extracts spatial features from coarse ERA5 fields
- bilinear upsampling moves features to PRISM grid size
- decoder predicts high-resolution temperature


In [ ]:
if not ERA5_PATH.exists():
    raise FileNotFoundError(f'Missing ERA5 file: {ERA5_PATH}')
if not PRISM_PATH.exists():
    raise FileNotFoundError(f'Missing PRISM directory: {PRISM_PATH}')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Missing checkpoint: {CHECKPOINT_PATH}')

dataset = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_PATH))
x0, y0 = dataset[0]
meta0 = dataset.metadata(0)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
cfg = checkpoint.get('model_config', {}) if isinstance(checkpoint, dict) else {}
model = CNNDownscaler(
    in_channels=cfg.get('in_channels', 1),
    out_channels=cfg.get('out_channels', 1),
    base_channels=cfg.get('base_channels', 32),
)
model.load_state_dict(checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint)
model = model.to(device).eval()

print('Aligned samples:', len(dataset))
print('First date:', meta0.date.date())
print('Input shape:', tuple(x0.shape), 'Target shape:', tuple(y0.shape))
print('Device:', device)


## 4) Inference / Pipeline Explanation
For one aligned date:
1. load ERA5 tensor
2. predict PRISM-resolution map using the CNN
3. compare prediction to PRISM target
4. compute sample RMSE and MAE


In [ ]:
date, x, y = dataset.metadata(0).date, *dataset[0]
x_batch = x.unsqueeze(0).to(device)
y_batch = y.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(x_batch, target_size=(y.shape[-2], y.shape[-1]))

sample_rmse = torch.sqrt(torch.mean((pred - y_batch) ** 2)).item()
sample_mae = torch.mean(torch.abs(pred - y_batch)).item()

print('Date:', date.date())
print(f'Sample RMSE: {sample_rmse:.4f}')
print(f'Sample MAE : {sample_mae:.4f}')


## 5) Results Visualization
This panel compares ERA5 input, model prediction, and PRISM target for the same date.


In [ ]:
era5_up = F.interpolate(
    x.unsqueeze(0),
    size=(y.shape[-2], y.shape[-1]),
    mode='bilinear',
    align_corners=False,
).squeeze(0)

era5_map = era5_up.squeeze().cpu().numpy()
pred_map = pred.squeeze(0).squeeze(0).cpu().numpy()
target_map = y.squeeze().cpu().numpy()

vmin = min(np.min(era5_map), np.min(pred_map), np.min(target_map))
vmax = max(np.max(era5_map), np.max(pred_map), np.max(target_map))

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
im = axes[0].imshow(era5_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0].set_title('ERA5 Input (Upsampled)')
axes[0].axis('off')

axes[1].imshow(pred_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1].set_title('CNN Prediction')
axes[1].axis('off')

axes[2].imshow(target_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[2].set_title('PRISM Target')
axes[2].axis('off')

fig.suptitle(f'ERA5 -> PRISM Downscaling | {date.date()}')
fig.colorbar(im, ax=axes, shrink=0.8, label='Temperature (deg C)')
plt.show()


## Saved Evaluation Figure
Load the saved output from the evaluation script when available.


In [ ]:
saved_plot = RESULTS_DIR / 'comparison_1_20230101.png'
if saved_plot.exists():
    display(Markdown('### Saved evaluation artifact'))
    display(Image(filename=str(saved_plot)))
else:
    cmd = ('python evaluation/evaluate_model.py --era5-path data_raw/era5_georgia_temp.nc '
           '--prism-path data_raw/prism --checkpoint-path checkpoints/cnn_downscaler_best.pt '
           '--num-samples 8 --num-plots 1 --results-dir results/evaluation')
    display(Markdown(f'Result image not found. Run:\n\n`{cmd}`'))


## Next Research Step
This CNN is a baseline. The next upgrade is a spatiotemporal model family (ConvLSTM / Transformer),
moving toward Prithvi WxC-style climate modeling directions.
